# Milestone 1: Set up the environment
Set up your own AWS account

# Milestone 2: Get Started

## Task 1: Download the Pinterest infrastructure
Here we set up the RDS database on locally on [DBeaver](https://dbeaver.io/) using MySQL, alternatively you can set this up on aws to run it on the cloud. We'll store the database credentials in a `local_db_creds.yaml` or `aws_db_creds.yaml` file depending on which environment you are using.

## Task 2: Sign in to the AWS console
Here we used our AWS IAM account to sign in to the AWS console. The working region is set to `eu-west-1` throughout this project.

## Configure our own MySQL database to host our data
We either set up our own local database or use a cloud database service like AWS RDS. Let's discuss both

### Setting up our own database locally

1. For macOS, we can use Homebrew to install MySQL:
    ```
    # Install via Homebrew
    brew install mysql

    # Start MySQL service
    brew services start mysql

    # Secure installation (set root password)
    mysql_secure_installation
    ```
2. Download (DBeaver Community Edition)[https://dbeaver.io]. Install and launch DBeaver.
3. Connect DBeaver to MySQL
    Open DBeaver → **Database** → **New Database Connection**.

    Select **MySQL** → Click **Next**.

    Configure the connection:
    
    **Host**: localhost

    **Port**: 3306

    **Username**: root

    **Password**: Enter the root password you set during MySQL installation.

    **Database**: Leave empty (create a new database later).

    Test the connection → Click **Finish**.

4. Create a New Database

    In DBeaver: Right-click your MySQL connection → **Create** → **Database**.
    Name the database (e.g., `pinterest_data_db`) → Click **OK**.

5. Import the SQL File

    Open the `pinterest_data_db` database in DBeaver.
    Right-click `pinterest_data_db` → Tools → Execute Script (or press Ctrl+Shift+X).
    Select your `pinterest_data_db.sql` file → Click **Start**.
    Wait for the script to execute. Check the Log tab for errors.

6. Verify the Import

    Expand the `pinterest_data_db` database → **Tables**.
    Right-click a table → **View Data** to confirm data exists.

**Other useful commands:**

To connect to mysql database run:
```
mysql -u root
```
To change the password of the root user run:
```
mysql -u root -p
ALTER USER 'root'@'localhost' IDENTIFIED BY 'new_password';
```
Please note that the username and password you set for the MySql database should be stored in `local_db_creds.yaml` for connnecting to the database later when running the `user_posting_emulation.py` file.

To start/stop/restart the mysql service run:
```
brew services start mysql
brew services stop mysql
brew services restart mysql
```

To see the status of all Homebrew services: 
```
brew services list
```

### TODO: Setting up our own database on AWS RDS

## Debugging MySQL Connection in DBeaver

- The error **"Public Key Retrieval is not allowed"** typically occurs when connecting to MySQL 8.0+ with certain security configurations. Here's how to fix it in DBeaver:

    Step 1: Edit Your MySQL Connection in DBeaver

    In DBeaver, right-click your MySQL connection → **Edit Connection**.
    Go to the **Connection Settings** tab.

    Step 2: Allow Public Key Retrieval

    Under the **Driver Properties** tab:

    Search for the property `allowPublicKeyRetrieval`.
    Set its value to `TRUE`.
    This bypasses the public key retrieval restriction for authentication.

# Milestone 3: Batch Processing: Configure the EC2 Kafka client

## Task 1: Create a .pem key file locally

If You Still Have the Original `.pem` Key when the EC2 was created, Use the existing key to SSH into the instance:
```
ssh -i "~/.ssh/mykeypair.pem" ubuntu@<your-ec2-public-ip>
```

If You’ve Lost the Original Key, Create a New Key and Add the New Public Key to the Instance:
1. On your local machine, create a new key file:
   ```
   ssh-keygen -y -f mykeypair.pem
   ```
2. Copy/Extract the Public Key from your new `.pem` file, then connect to the EC2 Instance via EC2 Instance Connect and Edit the `~/.ssh/authorized_keys` file:
    ```
    sudo nano ~/.ssh/authorized_keys
    ```
    Replace the existing public key with the new one (or add it as a new line if you want to keep both keys), save and exit `(Ctrl+O, Ctrl+X)`.

## Task 2: Connect to the EC2 instance

1. Here we use the `pinterest-ec2` instance on AWS for this project
2. [`Remote - SSH`](https://marketplace.visualstudio.com/items?itemName=ms-vscode-remote.remote-ssh) extension bug in vscode: 
    
    Shortly after starting up the EC2 instance, you may experience the issue of excessive CPU usage by the 'rg' or/and the 'node' process in VSCode (run `top` to monitor CPU usage on the EC2 instance). This can be resolved by first killing the 'rg' or/and the 'node' processes (`kill -9 <rg_PID> <node_PID>`), then setting `"search.followSymlinks"` to false in VSCode Settings: `Command + , (macOS shorcut)`/`Ctrl + , (Windows shorcut)` -> `Search Settings (Settings.json)` -> add this line `"search.followSymlinks": false`. After that, restart the EC2 instance and ssh to it. See [GitHub Issue #98594](https://github.com/microsoft/vscode/issues/98594) for more information.

3. Setting up Elastic IP on AWS: To retain the same public IP address and DNS name after restarts, use AWS Elastic IP (EIP): Go to `EC2 Dashboard` → `Elastic IPs` → `Allocate Elastic IP address`. Then, select the Elastic IP → `Action` → `Associate Elastic IP address` → Choose your EC2 instance and click `Associate` .
4. Final config in `~/.ssh/config`:
    ```
    ###########################################
    ########### pinterest-ec2 Login ###########
    ###########################################
    Host aws-pinterest-ec2
        HostName <your_aws_ec2_elastic_ip>
        User ubuntu
        IdentityFile ~/.ssh/mykeypair.pem
    ```

## Task 3: Create Kafka Topics on EC2

Find your UserId/Account ID using the AWS CLI:
```
aws sts get-caller-identity
```
Under the "Account" section, you will find your Account ID. Here are our three Kafka topics to create:
```
<account_ID>.pin for the Pinterest posts data 
<account_ID>.geo for the post geolocation data
<account_ID>.user for the post user data

```
(where account_ID = your_UserId)

<!-- topics=808492447622.pin,808492447622.geo,808492447622.user -->

**All kafka related services are configured as services which will start automatically on startup.** Here are the main services we'll be using:
```
zookeeper.service
kafka-server.service
kafka-rest.service
kafka-connect.service
schema-registry.service
```

Create a Kafka topic:
```
kafka-topics --create \
  --bootstrap-server localhost:9092 \
  --replication-factor 1 \
  --partitions 3 \
  --topic <kafka-test-topic>
```

List all the Kafka topics that are currently available on the Kafka broker running at localhost:9092
```
kafka-topics --list \
  --bootstrap-server localhost:9092
```

Verify with kafka-console-consumer on EC2 instance:
```
kafka-console-consumer --bootstrap-server localhost:9092 --topic <account_ID>.pin  --from-beginning
```

Delete a topic:
```
kafka-topics --bootstrap-server localhost:9092 --delete --topic <account_ID>.pin
```
Confirm a topic exist and has data:
```
kafka-topics --describe --bootstrap-server localhost:9092 --topic <account_ID>.pin
```
Restart the Connector Service and check logs
```
sudo systemctl daemon-reload
sudo systemctl restart kafka-connect.service
journalctl -u kafka-connect.service -f
# curl -X POST http://localhost:8083/connectors/s3-sink/restart
```
Monitor the connector status again with:
```
curl http://localhost:8083/connectors/s3-sink/status
```
Test if the EC2 instance can access the S3 bucket using the AWS CLI:
```
aws s3 ls s3://<your_bucket_name>
```

**_Example usage for this project:_**
1. Create the Kafka Topics (pin, geo, user)
  ```
  kafka-topics --create --topic <your_UserId>.pin --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
  kafka-topics --create --topic <your_UserId>.geo --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
  kafka-topics --create --topic <your_UserId>.user --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
  ```
2. Verify the Kafka Topics
  ```
  kafka-topics --list --bootstrap-server localhost:9092
  ```
3. (Optional) Describe the Topics
  ```
  kafka-topics --describe --topic <your_UserId>.pin --bootstrap-server localhost:9092
  kafka-topics --describe --topic <your_UserId>.geo --bootstrap-server localhost:9092
  kafka-topics --describe --topic <your_UserId>.user --bootstrap-server localhost:9092
  ```


# Milestone 4: Batch Processing: Configuring an API in API Gateway

## Task 1: Build a Kafka REST proxy integration method for the API

Follow the instructions in the `2. Integrating API Gateway with Kafka.ipynb`
notebook to complete this part.
Keys things to remember:
- *Security Groups: Ensure your EC2 security group allows inbound traffic on:*
    ```
    2181 (ZooKeeper)
    9092 (Kafka Server)
    8081 (Schema Registry)
    8082 (Kafka REST Proxy)
    8083 (Kafka Connect)
    ```
- *IAM Role: Verify the EC2 instance has permissions to access S3 (if using Kafka Connect S3 sink)*: Here `EC2-S3FullAccess` was used for the IAM Role.
  

## Edite java properties files:

The file `/home/ubuntu/kafka/etc/kafka/s3-sink.properties` is a configuration file for the Confluent S3 Sink Connector, which is part of Kafka Connect. This connector is used to export data from Apache Kafka topics to Amazon S3 in a structured format (e.g., JSON, Avro). Here we need to configure the following properties:
```
topics=<account_ID>.pin,<account_ID>.geo,<account_ID>.user 
s3.region=<your_s3_region> #Set your AWS region
s3.bucket.name=<your_pinterest_confluent_kafka_connect_s3> #Set your S3 bucket name
format.class=io.confluent.connect.s3.format.json.JsonFormat
```

For more information, check [Amazon S3 Source Connector for Confluent Cloud](https://docs.confluent.io/cloud/current/connectors/cc-s3-source.html#using-the-confluent-cli) for using the connector

Here we only need to configure three properties files: `s3-sink.properties`, `server.properties` and `kafka-rest.properties`. And below are some of the important properties that we need to configure:

- For Socket Server Settings In `server.properties`:
    ```
    listeners=PLAINTEXT://0.0.0.0:9092
    advertised.listeners=PLAINTEXT://localhost:9092
    ```
- For S3 Sink Connector Settings In `s3-sink.properties`:
    ```
    # Replace PLACEHOLDER with your UserId
    topics=PLACEHOLDER.pin,PLACEHOLDER.geo,PLACEHOLDER.user
    # Choose the correct region for your S3 bucket
    s3.region=eu-west-1
    # Put the name of your S3 bucket here
    s3.bucket.name=pinterest-confluent-kafka-connect-s3
    format.class=io.confluent.connect.s3.format.json.JsonFormat
    ```
- In `kafka-rest.properties`
    ```
    schema.registry.url=http://localhost:8081
    zookeeper.connect=http://localhost:2181
    bootstrap.servers=PLAINTEXT://localhost:9092
    ```
As a side note, make sure `zookeeper.properties` is correctly configured as well.

If we run `sudo systemctl status kafka-connect.service` then we can see that we're running the `connect-standalone` bin command, using the `connect-standalone.properties` and `s3-sink.properties` files for the configured properties.
```
[Unit]
Description=Apache Kafka Connect - distributed
Documentation=http://docs.confluent.io/
After=network.target kafka-server.service

[Service]
Type=simple
ExecStart=/home/ubuntu/kafka/bin/connect-standalone /home/ubuntu/kafka/etc/kafka/connect-standalone.properties /home/ubuntu/kafka/etc/kafka/s3-sink.properties
TimeoutStopSec=180
Restart=no

Environment="KAFKA_OPTS=-Dcom.amazonaws.sdk.debug=all"

[Install]
WantedBy=multi-user.target
```

## Run Kafka related services

After you've edited the Java properties files, you can run the following command to restart the services for the changes to take effect:
```
sudo systemctl stop zookeeper.service
sudo systemctl stop kafka-server.service
sudo systemctl stop kafka-rest.service
sudo systemctl stop kafka-connect.service
sudo systemctl stop schema-registry.service

sudo systemctl start zookeeper.service
sudo systemctl start kafka-server.service
sudo systemctl start kafka-rest.service
sudo systemctl start kafka-connect.service
sudo systemctl start schema-registry.service
```
Alternatively albeit less reliable:
```
sudo systemctl restart zookeeper.service
sudo systemctl restart kafka-server.service
sudo systemctl restart kafka-rest.service
sudo systemctl restart kafka-connect.service
sudo systemctl restart schema-registry.service
```
To view the status of the restarted services:
```
sudo systemctl status zookeeper.service
sudo systemctl status kafka-server.service
sudo systemctl status kafka-rest.service
sudo systemctl status kafka-connect.service
sudo systemctl status schema-registry.service
```

Other useful commands systemctl commands:

List all active systemd units related to Zookeeper, Kafka, Kafka Server, and Kafka Connect by filtering the output of systemctl list-units for each respective service:
```
systemctl list-units  | grep zookeeper
systemctl list-units  | grep kafka
systemctl list-units  | grep kafka-server
systemctl list-units  | grep kafka-connect
```


## Task 2: Send data to API Gateway

- Expected output from running `curl http://localhost:8083/connectors/s3-sink/status`:
    ```
    ubuntu@ip-172-31-36-151:~$ curl http://localhost:8083/connectors/s3-sink/status
    {"name":"s3-sink","connector":{"state":"RUNNING","worker_id":"172.31.36.151:8083"},"tasks":[{"id":0,"state":"RUNNING","worker_id":"172.31.36.151:8083"}],"type":"sink"}
    ```
- Expected output from running `journalctl -u kafka-rest.service -f`:
    ```
    ubuntu@ip-172-31-36-151:~$ journalctl -u kafka-rest.service -f
    Mar 10 17:38:00 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:00,994] INFO Started o.e.j.s.ServletContextHandler@595f4da5{/,null,AVAILABLE} (org.eclipse.jetty.server.handler.ContextHandler:921)
    Mar 10 17:38:01 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:01,257] INFO Started o.e.j.s.ServletContextHandler@42561fba{/ws,null,AVAILABLE} (org.eclipse.jetty.server.handler.ContextHandler:921)
    Mar 10 17:38:01 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:01,439] INFO Started NetworkTrafficServerConnector@65b104b9{HTTP/1.1, (http/1.1, h2c)}{0.0.0.0:8082} (org.eclipse.jetty.server.AbstractConnector:333)
    Mar 10 17:38:01 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:01,441] INFO Started @31699ms (org.eclipse.jetty.server.Server:415)
    Mar 10 17:38:01 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:01,442] INFO Server started, listening for requests... (io.confluent.kafkarest.KafkaRestMain:48)
    Mar 11 01:52:31 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 01:52:31,112] INFO 159.89.54.154 - - [11/Mar/2025:01:52:30 +0000] "GET / HTTP/1.1" 200 22 "-" "Mozilla/5.0 (compatible)" 392 - (io.confluent.rest-utils.requests:62)
    Mar 11 01:52:31 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 01:52:31,428] INFO 159.89.54.154 - - [11/Mar/2025:01:52:31 +0000] "GET /favicon.ico HTTP/1.1" 404 5133 "http://18.200.92.19:8082/" "Mozilla/5.0 (compatible)" 83 - (io.confluent.rest-utils.requests:62)
    Mar 11 05:15:03 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 05:15:03,177] INFO 162.142.125.126 - - [11/Mar/2025:05:15:03 +0000] "GET / HTTP/1.1" 200 22 "-" "Mozilla/5.0 (compatible; CensysInspect/1.1; +https://about.censys.io/)" 5 - (io.confluent.rest-utils.requests:62)
    Mar 11 05:15:05 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 05:15:05,186] INFO 162.142.125.126 - - [11/Mar/2025:05:15:05 +0000] "GET /favicon.ico HTTP/1.1" 404 5133 "-" "Mozilla/5.0 (compatible; CensysInspect/1.1; +https://about.censys.io/)" 4 - (io.confluent.rest-utils.requests:62)
    Mar 11 05:15:09 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 05:15:09,086] INFO 162.142.125.126 - - [11/Mar/2025:05:15:09 +0000] "GET /favicon.ico HTTP/1.1" 404 5133 "-" "Mozilla/5.0 (compatible; CensysInspect/1.1; +https://about.censys.io/)" 8 - (io.confluent.rest-utils.requests:62)
    ```

### Debugging

Issues I had when running this part of the project:
1. The specified bucket is not valid.
    ```
    • ubuntu@ip-172-31-36-151:~$ curl http://localhost: 8083/connectors/s3-sink/status
    {"name": "s3-sink", "connector" : {"state" : "RUNNING"
    ', "worker_id": "172.31.36.151:8083"}, "tasks": [{"id" :0, "state": "FAILED", "worker_id": "172.31
    -36.151:8083"
    ',"trace": "org-apache.kafka. connect.errors.ConnectException: com.amazonaws.services.s3.model.AmazonS3Exception: The specifie
    d bucket is not valid. (Service: Amazon S3; Status Code: 400; Error Code: InvalidBucketName; Request ID: 3EPCDZ7G5VXEH0J7; S3 Extended R equest ID: BdpY/Y/9wUWQ1qMEexyGqVtMuMzgHb0EMdy2sV/7a2zl0GfjJuFYbuNr26Ch72G0YaTd+E1WlYF59KyVPicz2iNKDkd0VRTRQ3kn40o0N2Q=; Proxy: nutt),
    3 Extended Request ID: BdpY/Y/9wUWQ1qMЕexyGqVtMuMzgHb0EMdy2sV/7a2zl0GfjJuFYbuNr26Ch72G0YaTd+E1WlYF59KyVPicz2iNKDkd0VRTRQ3kn40o0N2Q=\n\ta
    t io.confluent.connect.s3.S3SinkTask.start(S3SinkTask,java:142)\n\tat org.apache.kafka.connect.runtime.WorkerSinkTask.initializeAndStart
    (WorkerSinkTask.java:333)\n\tat org-apache.kafka. connect. runtime.WorkerTask.doRun(WorkerTask.java:227)\n\tat org-apache.kafka. connect.ru
    ntime.WorkerTask. run (WorkerTask.java: 284)\n\tat
    org.apache.kafka.connect.runtime.isolation.Plugins.lambda$withClassLoader$7(Plugins-java
    :339)\n\tat java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)\n\tat java.base/java.util.concurrent.Futur
    eTask. run (FutureTask.java:264)\n\tat java.base/java.util.concurrent.ThreadPoolExecutor. runWorker(ThreadPoolExecutor-java:1128)\n\tat jav a.base/java.util.concurrent.ThreadPoolExecutor$Worker. run(ThreadPoolExecutor.java:628)\n\tat java.base/java.lang.Thread. run(Thread-java:
    829) \nCaused by: com.amazonaws.services.s3.model.AmazonS3Exception: The specified bucket is not valid. (Service: Amazon S3; Status Code:
    400; Error Code: InvalidBucketName; Request ID: 3EPCDZ7G5VXEH0J7; S3 Extended Request ID: BdpY/Y/9wUWQ1qMЕexyGqVtMuMzgHb0EMdy2sV/7a2z10
    GfjJuFYbuNr26Ch72G0YaTd+E1W1YF59KyVPicz2iNKDkd0VRTRQ3kn40o0N2Q=; Proxy: null), S3 Extended Request ID: BdpY/Y/9wUWQ1qMEexyGqVtMuMzgHb0EM
    dy2sV/7a2z10GfjJuFYbuNr26Ch72G0YaTd+E1WlYF59KyVPicz2iNKDkd0VRTRQ3kn40o0N2Q=\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor.ha
    ndleErrorResponse(AmazonHttpClient.java: 1879)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor.handleServiceErrorResponse (Amazo
    nHttpClient. java: 1418)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor-executeOneRequest(AmazonHttpClient.java:1387)\n\tat com
    .amazonaws.http.AmazonHttpClient$RequestExecutor.executeHelper(AmazonHttpClient.java:1157)\n\tat com.amazonaws.http.AmazonHttpClient$Req
    uestExecutor.doExecute(AmazonHttpClient.java:814)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor.executeWithTimer(AmazonHttpC
    lient. java:781)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor.execute(AmazonHttpClient.java:755)\n\tat com.amazonaws.http.Am
    azonHttpClient$RequestExecutor.access$500(AmazonHttpClient.java:715)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutionBuilderIm
    pl.execute(AmazonHttpClient.java:697)\n\tat com.amazonaws.http.AmazonHttpClient.execute(AmazonHttpClient.java:561)\n\tat com.amazonaws.h
    ttp.AmazonHttpClient.execute(AmazonHttpClient.java:541)\n\tat com.amazonaws.services.s3.AmazonS3Client.invoke(AmazonS3Client.java:5520)\
    n\tat com.amazonaws.services.s3.AmazonS3Client. invoke(AmazonS3Client.java:5467)\n\tat com.amazonaws.services.s3.AmazonS3Client.getAcl(Am
    azonS3Client. java: 4113)\n\tat com.amazonaws.services.s3.AmazonS3Client.getBucketAcl(AmazonS3Client.java:1308)\n\tat com.amazonaws.servic
    es. s3.AmazonS3Client.getBucketAcl(AmazonS3Client.java:1298)\n\tat com.amazonaws.services.s3.AmazonS3Client.doesBucketExistV2(AmazonS3Cli
    ent. java: 1436)\n\tat io.confluent.connect.s3.storage.S3Storage.bucketExists(S3Storage.java:186)\n\tat io.confluent.connect.s3.53SinkTask
    ```
    The reason for the error was because in java property files, a logical line cannot be followed by a comment line in the line. See example below from the `s3-sink.properties` file:
    ```
    s3.bucket.name=pinterest-confluent-kafka-connect-s3 # Replace this with your s3 bucket name
    ```
    The above line will not work as the element for the key (`s3.bucket.name`) will be treated as a whole (`pinterest-confluent-kafka-connect-s3 # Replace this with your s3 bucket name`) which is not what we want. See [java Properties Class -> load()](https://docs.oracle.com/javase/7/docs/api/java/util/Properties.html) for more info. The correct syntax should be:
    ```
    # Replace this with your s3 bucket name
    s3.bucket.name=pinterest-confluent-kafka-connect-s3
    ```
2. For this line `log4j.appender.connectAppender.layout=org.apache.log4j.PatternLayout` in file `connect-log4j.properties`, the `l` in `log4j.appender.connectAppender.layout` was origianlly written as a uppercase `L` instead of lowercase `l` which caused the error.

Other useful commands:

To view the logs for the `kafka-connect.service` in reverse chronological order:
```
journalctl -u kafka-connect.service -r
```
Manages cron jobs in edit mode, which are scheduled tasks that run automatically at specified intervals.
```
crontab -e
```


# Milestone 5: Batch Processing: Databricks

## Configure Databricks to allow permissions to read in data from S3.

Go to `Pinterest Data Engineering Project` workspace homepage on Databricks. Click on `+ New` on top left -> `Add or upload data` -> `Create table from Amazon S3` -> Click on existing external s3 location -> Click on `+`(Create new) -> Under `Create a new external location` popup -> slect `AWS Quickstart (Recommended)` -> `Next` -> Bucket Name: `s3://pinterest-confluent-kafka-connect-s3/` -> Personal Access Token: Click on `Generate new token` -> `Launch in Quickstart` -> copy the generated token and paste into the 'Databricks Personal Access Token' field of the CloudFormation template -> `Create stack`. Then wait for `CREATE_COMPLETE` status to show up in CloudFormation > Stacks > databricks-s3-ingest-xxxxx. You can click on template while it's doing it to see what's actually going on. This is very similar to [ARM](https://learn.microsoft.com/en-us/azure/azure-resource-manager/management/overview) in Azure in that they both provide infrastructure as code (IaC) capabilities for managing cloud resources. After this is done, you should be able to see the data in your s3 bucket by going to workspace homepage on Databricks -> Click on `+ New` on top left -> `Add or upload data` -> `Create table from Amazon S3`, and under 
`Select all` you will see the data in the external s3 location.

## Task 2: Read data from the S3 bucket to Databricks

In [0]:
# Read all partitions for all three topics
df_pin = spark.read.json("s3a://pinterest-confluent-kafka-connect-s3/topics/808492447622.pin/partition=*/")
df_geo = spark.read.json("s3a://pinterest-confluent-kafka-connect-s3/topics/808492447622.geo/partition=*/")
df_user = spark.read.json("s3a://pinterest-confluent-kafka-connect-s3/topics/808492447622.user/partition=*/")

# Verify the combined data
# df_pin.show(10)
display(df_pin.head(5))
display(df_geo.head(5))
display(df_user.head(5))

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
art,"Framed Canvas Home Artwork Decoration Abstract Mountain Nature Scenery Canvas Wall Art for Living Room, Bedroom Canvas Wall Art Ready to Hang Each canvas is professionally print…",1,305,https://i.pinimg.com/originals/ae/fa/69/aefa69dc49f836f15a6a1d0322c9b012.jpg,683,image,Wall Canvas Mall,Local save in /data/art,"Bedroom Canvas,Canvas Home,Canvas For Living Room,Paintings For Living Room,Living Room Gallery Wall,Living Room Wall Art,Wall Art Bedroom,Yellow Walls Living Room,Modern Gallery Wall",Framed Canvas Home Artwork Decoration Abstract Mountain Nature Scenery Canvas Wall Art for Living Room Bedroom Canvas Wall Art Ready to Hang - Unframed 1P 24x18 / Gallery Wrap,a0215254-abdd-4fbf-8d79-927aa319587e
christmas,"Make your home look festive for Christmas on a budget with these DIY dollar store Christmas decorations. From DIY Christmas wreaths to DIY Christmas centerpieces, there are plen…",1,647k,https://i.pinimg.com/videos/thumbnails/originals/1c/a4/a7/1ca4a7bb545354dbd8c65a27c1b920b7.0000001.jpg,1659,video,"Prudent Penny Pincher - Home Decor, Organization, Crafts, Recipes",Local save in /data/christmas,"Dollar Store Christmas,Christmas On A Budget,Christmas Gifts For Mom,Diy Christmas Decorations Easy,Diy Christmas Ornaments,Outdoor Christmas,Christmas Arts And Crafts,Christmas Gift Baskets,Xmas Crafts",100 DIY Dollar Store Christmas Decor Ideas,7d27a02b-2708-4ba1-bb73-24577deafc2e
education,sight words activities center plus additional ideas for revving up your sight words practice #sightwords #sightwordrecognition #sightwordsactivities,1,23k,https://i.pinimg.com/originals/e2/47/cf/e247cf9bf377e4ac4a54e9f24b6bacd6.jpg,3571,image,Lessons for Little Ones by Tina O'Block,Local save in /data/education,"Kindergarten Reading,Kindergarten Activities,Educational Activities,Center Ideas For Kindergarten,Fine Motor Activities For Kids,Kindergarten Sight Words,Jolly Phonics Activities,Kindergarten Classroom Organization,Kindergarten Morning Work",Ideas for Revving Up Sight Words Practice - Lessons for Little Ones by Tina O'Block,6d61aacc-dc1a-4eaa-839d-49fab3721cce
christmas,Brand Name: PIGZOOUse: PaintingsOrigin: CN(Origin)External Packaging: Colored BoxForm: Singleis_customized: YesMaterial: ResinPasting Area: FullType of Wholesale: noPattern Type…,1,1k,https://i.pinimg.com/originals/e5/80/09/e580093eb847d9e829cb5bb7a4afb1ea.jpg,1809,image,Darrechi,Local save in /data/christmas,"Christmas Scenery,Christmas Red Truck,Christmas Mood,Noel Christmas,Christmas Pictures,Christmas Greetings,Christmas Crafts,Vintage Christmas Photos,Christmas Animals",5D DIY Diamond Painting Kit Christmas animals Car donkey Dog Full Square&Round embroidery mosaic Cross stitch Paint home decor - 3 / Round Drill 45X60,c90cfe22-7c3e-4324-845c-8ec792d8046d
christmas,Add a personal touch to your Christmas ornaments this year with these creative and festive ideas DIY Christmas ornaments. These beautifully decorated homemade Christmas ornament…,1,647k,https://i.pinimg.com/videos/thumbnails/originals/5d/19/50/5d1950fc54cff265d72c8e285081206c.0000001.jpg,1884,video,"Prudent Penny Pincher - Home Decor, Organization, Crafts, Recipes",Local save in /data/christmas,"Christmas Tree Decorations For Kids,Homade Christmas Ornaments,Homemade Christmas Decorations,Christmas Gift Baskets,Christmas Crafts For Kids,Christmas Diy,Christmas Bulbs,Fun Diy Crafts,Diy Ornaments",200 DIY Christmas Ornaments,48e9116f-48bc-4f0d-9b32-9b724db774e9


country,ind,latitude,longitude,timestamp
British Indian Ocean Territory (Chagos Archipelago),8221,-20.5574,-54.4834,2021-12-29T06:33:46
Antarctica (the territory South of 60 deg S),1910,-88.4642,-171.061,2020-06-14T01:25:40
Antarctica (the territory South of 60 deg S),2073,-88.4642,-171.061,2021-05-21T04:24:58
Antarctica (the territory South of 60 deg S),2295,-88.4642,-171.061,2020-09-17T10:30:11
Antarctica (the territory South of 60 deg S),2587,-39.3929,-177.136,2019-07-09T21:57:35


age,date_joined,first_name,ind,last_name
38,2015-11-12T11:07:24,Christopher,2814,Hernandez
25,2016-11-24T18:36:05,Christopher,3587,Rodriguez
24,2016-04-06T21:17:29,Christopher,8221,Edwards
45,2016-09-15T07:02:53,Christopher,9582,Hawkins
46,2016-07-02T09:06:40,Christina,1683,Carpenter


Ensure all index columns match exactly in all three DataFrames (df_pin, df_geo & df_user)

In [0]:
from pyspark.sql.functions import col

# Collect the index values from each DataFrame using their respective column names
index_pin = df_pin.select("index").distinct().select(col("index")).collect()
index_geo = df_geo.select("ind").distinct().select(col("ind")).collect()
index_user = df_user.select("ind").distinct().select(col("ind")).collect()

# Extract the index values from the collected rows
index_pin_values = [row["index"] for row in index_pin]
index_geo_values = [row["ind"] for row in index_geo]
index_user_values = [row["ind"] for row in index_user]

# Compare the index values
def compare_indexes(index1, index2, index3):
    # Convert lists to sets for comparison
    set1 = set(index1)
    set2 = set(index2)
    set3 = set(index3)
    
    # Check if all sets are equal
    return set1 == set2 == set3

# Perform the comparison
match_result = compare_indexes(index_pin_values, index_geo_values, index_user_values)

# Print the result
if match_result:
    print("All index columns match exactly!")
else:
    print("Index columns do not match.")

# Alternatively print out all three DataFrames, sort the index in ascending/descending order and manually inpect some sample index values to make sure they match across different DataFrames.
# display(df_pin)
# display(df_geo)
# display(df_user)

All index columns match exactly!


Check if all values in `downloaded` are the same

In [0]:
# Sum up all the values in the "downloaded" column
from pyspark.sql.functions import col, sum
total_sum = df_pin.select(sum(col("downloaded"))).collect()[0][0]

# Print the total sum
print(f"The total sum of the 'downloaded' column is: {total_sum}")
# The result might show not all downloaded values are 1.

The total sum of the 'downloaded' column is: 491


In [0]:
# Print the number of rows
num_rows = df_pin.count()
num_rows = df_geo.count()
num_rows = df_user.count()

print(f'The number of rows in the df_pin is: {num_rows}')
print(f'The number of rows in the df_geo is: {num_rows}')
print(f'The number of rows in the df_user is: {num_rows}')

# Print the schema of the three DataFrames
print("Schema for df_pin:")
df_pin.printSchema()
print("Schema for df_geo:")
df_geo.printSchema()
print("Schema for df_user:")
df_user.printSchema()

The number of rows in the df_pin is: 500
The number of rows in the df_geo is: 500
The number of rows in the df_user is: 500
Schema for df_pin:
root
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- downloaded: long (nullable = true)
 |-- follower_count: string (nullable = true)
 |-- image_src: string (nullable = true)
 |-- index: long (nullable = true)
 |-- is_image_or_video: string (nullable = true)
 |-- poster_name: string (nullable = true)
 |-- save_location: string (nullable = true)
 |-- tag_list: string (nullable = true)
 |-- title: string (nullable = true)
 |-- unique_id: string (nullable = true)

Schema for df_geo:
root
 |-- country: string (nullable = true)
 |-- ind: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timestamp: string (nullable = true)

Schema for df_user:
root
 |-- age: long (nullable = true)
 |-- date_joined: string (nullable = true)
 |-- first_name: string (nullab

# Milestone 6: Batch Processing: Spark on Databricks

## Task 1: Clean the DataFrame that contains information about Pinterest posts.

In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace, expr

# Replace empty entries and entries with no relevant data in string columns with None
string_columns = [col_name for col_name, dtype in df_pin.dtypes if dtype == 'string']
for column in string_columns:
    df_pin = df_pin.withColumn(
        column, 
        when(trim(col(column)) != "", col(column)).otherwise(None)
    )
# Replace invalid or missing entries in numeric columns with None
numeric_columns = [col_name for col_name, dtype in df_pin.dtypes if dtype in ['int', 'double', 'long']]
for column in numeric_columns:
    df_pin = df_pin.withColumn(
        column, 
        when(col(column).isNotNull(), col(column)).otherwise(None)
    )

# Transform follower_count to integer with multiple suffix cases.
df_pin = df_pin.withColumn(
    "follower_count",
    # Regex replacements to handle suffixes like k (thousand), M (million), B (billion). (?i)" makes the pattern case-insensitive while '$' anchors the match to the end of the string.
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("follower_count"), 
                "(?i)k$", "000"                                          # 1k -> 1000
            ),
            "(?i)m$", "000000"                                           # 1M -> 1000000
        ),
        "(?i)b$", "000000000"                                            # 1B -> 1000000000
    )
)

# Filter rows where follower_count does not match a valid integer pattern
df_invalid_entries = df_pin.filter(
    ~col("follower_count").rlike(r"^\d+$")  # Negation of regex for valid integers, Use raw string (r"...")
)


# Show all rows with invalid follower_count entries
print("Invalid follower_count entries:")
display(df_invalid_entries)

# Clean follower_count to handle string values like "User Info Error"
df_pin = df_pin.withColumn(
    "follower_count",
    expr("try_cast(follower_count as int)")  # Tolerate invalid entries by converting them to NULL
)

# Ensure numeric columns are correctly typed
numeric_columns = ["downloaded", "follower_count", "index"]
for column in numeric_columns:
    df_pin = df_pin.withColumn(
        column,
        col(column).cast("int")
    )

# Clean save_location column to keep only the path by removing the prefix "Local save in " from its values
df_pin = df_pin.withColumn(
    "save_location",
    regexp_replace(col("save_location"), "^Local save in ", "") # '^' anchors the match to the start of the string.
)

# Rename index column to ind
df_pin = df_pin.withColumnRenamed("index", "ind")

# Reorder columns
# desired_order = [
#     "ind", "unique_id", "title", "description", "follower_count",
#     "downloaded", "poster_name", "tag_list", "is_image_or_video", 
#     "image_src", "save_location", "category"
# ]
desired_order = [
    "ind", "unique_id", "title", "description", "follower_count", 
    "poster_name", "tag_list", "is_image_or_video", 
    "image_src", "save_location", "category"
]
df_pin_cleaned = df_pin.select(desired_order)

Invalid follower_count entries:


category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
diy-and-crafts,No description available Story format,0,User Info Error,Image src error.,2898,multi-video(story page format),User Info Error,Local save in /data/diy-and-crafts,"N,o, ,T,a,g,s, ,A,v,a,i,l,a,b,l,e",No Title Data Available,b0ec6a4e-2a0c-4608-891d-7041bb29be10
travel,No description available Story format,0,User Info Error,Image src error.,9966,multi-video(story page format),User Info Error,Local save in /data/travel,"N,o, ,T,a,g,s, ,A,v,a,i,l,a,b,l,e",No Title Data Available,0ee39bf8-f1e7-4e7d-af87-c4e6dc04bc62


In [0]:
# Display the first 5 rows of the cleaned DataFrame "df_pin_cleaned" to inspect the data
display(df_pin_cleaned.head(5))

# Count the total number of rows in the cleaned DataFrame "df_pin_cleaned"
num_rows = df_pin_cleaned.count()

# Print the total number of rows after cleaning the DataFrame
print(f'The number of rows after cleaning df_pin in Task 1 is: {num_rows}')

# Print the schema of the cleaned DataFrame `df_pin` to verify column names and data types
df_pin_cleaned.printSchema()

ind,unique_id,title,description,follower_count,poster_name,tag_list,is_image_or_video,image_src,save_location,category
683,a0215254-abdd-4fbf-8d79-927aa319587e,Framed Canvas Home Artwork Decoration Abstract Mountain Nature Scenery Canvas Wall Art for Living Room Bedroom Canvas Wall Art Ready to Hang - Unframed 1P 24x18 / Gallery Wrap,"Framed Canvas Home Artwork Decoration Abstract Mountain Nature Scenery Canvas Wall Art for Living Room, Bedroom Canvas Wall Art Ready to Hang Each canvas is professionally print…",305,Wall Canvas Mall,"Bedroom Canvas,Canvas Home,Canvas For Living Room,Paintings For Living Room,Living Room Gallery Wall,Living Room Wall Art,Wall Art Bedroom,Yellow Walls Living Room,Modern Gallery Wall",image,https://i.pinimg.com/originals/ae/fa/69/aefa69dc49f836f15a6a1d0322c9b012.jpg,/data/art,art
1659,7d27a02b-2708-4ba1-bb73-24577deafc2e,100 DIY Dollar Store Christmas Decor Ideas,"Make your home look festive for Christmas on a budget with these DIY dollar store Christmas decorations. From DIY Christmas wreaths to DIY Christmas centerpieces, there are plen…",647000,"Prudent Penny Pincher - Home Decor, Organization, Crafts, Recipes","Dollar Store Christmas,Christmas On A Budget,Christmas Gifts For Mom,Diy Christmas Decorations Easy,Diy Christmas Ornaments,Outdoor Christmas,Christmas Arts And Crafts,Christmas Gift Baskets,Xmas Crafts",video,https://i.pinimg.com/videos/thumbnails/originals/1c/a4/a7/1ca4a7bb545354dbd8c65a27c1b920b7.0000001.jpg,/data/christmas,christmas
3571,6d61aacc-dc1a-4eaa-839d-49fab3721cce,Ideas for Revving Up Sight Words Practice - Lessons for Little Ones by Tina O'Block,sight words activities center plus additional ideas for revving up your sight words practice #sightwords #sightwordrecognition #sightwordsactivities,23000,Lessons for Little Ones by Tina O'Block,"Kindergarten Reading,Kindergarten Activities,Educational Activities,Center Ideas For Kindergarten,Fine Motor Activities For Kids,Kindergarten Sight Words,Jolly Phonics Activities,Kindergarten Classroom Organization,Kindergarten Morning Work",image,https://i.pinimg.com/originals/e2/47/cf/e247cf9bf377e4ac4a54e9f24b6bacd6.jpg,/data/education,education
1809,c90cfe22-7c3e-4324-845c-8ec792d8046d,5D DIY Diamond Painting Kit Christmas animals Car donkey Dog Full Square&Round embroidery mosaic Cross stitch Paint home decor - 3 / Round Drill 45X60,Brand Name: PIGZOOUse: PaintingsOrigin: CN(Origin)External Packaging: Colored BoxForm: Singleis_customized: YesMaterial: ResinPasting Area: FullType of Wholesale: noPattern Type…,1000,Darrechi,"Christmas Scenery,Christmas Red Truck,Christmas Mood,Noel Christmas,Christmas Pictures,Christmas Greetings,Christmas Crafts,Vintage Christmas Photos,Christmas Animals",image,https://i.pinimg.com/originals/e5/80/09/e580093eb847d9e829cb5bb7a4afb1ea.jpg,/data/christmas,christmas
1884,48e9116f-48bc-4f0d-9b32-9b724db774e9,200 DIY Christmas Ornaments,Add a personal touch to your Christmas ornaments this year with these creative and festive ideas DIY Christmas ornaments. These beautifully decorated homemade Christmas ornament…,647000,"Prudent Penny Pincher - Home Decor, Organization, Crafts, Recipes","Christmas Tree Decorations For Kids,Homade Christmas Ornaments,Homemade Christmas Decorations,Christmas Gift Baskets,Christmas Crafts For Kids,Christmas Diy,Christmas Bulbs,Fun Diy Crafts,Diy Ornaments",video,https://i.pinimg.com/videos/thumbnails/originals/5d/19/50/5d1950fc54cff265d72c8e285081206c.0000001.jpg,/data/christmas,christmas


The number of rows after cleaning df_pin in Task 1 is: 500
root
 |-- ind: integer (nullable = true)
 |-- unique_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- follower_count: integer (nullable = true)
 |-- poster_name: string (nullable = true)
 |-- tag_list: string (nullable = true)
 |-- is_image_or_video: string (nullable = true)
 |-- image_src: string (nullable = true)
 |-- save_location: string (nullable = true)
 |-- category: string (nullable = true)



## Task 2: Clean the DataFrame that contains information about geolocation.

In [0]:
from pyspark.sql.functions import array, col, to_timestamp

# Create a new column "coordinates" as an array of latitude and longitude
df_geo = df_geo.withColumn("coordinates", array(col("latitude"), col("longitude")))

# Drop the latitude and longitude columns
df_geo = df_geo.drop("latitude", "longitude")

# Convert the "timestamp" column from string to timestamp
df_geo = df_geo.withColumn("timestamp", to_timestamp(col("timestamp")))

# Reorder the columns
df_geo_cleaned = df_geo.select("ind", "country", "coordinates", "timestamp")

# Show the cleaned DataFrame schema
print("Schema for df_geo_cleaned:")
df_geo_cleaned.printSchema()

Schema for df_geo_cleaned:
root
 |-- ind: long (nullable = true)
 |-- country: string (nullable = true)
 |-- coordinates: array (nullable = false)
 |    |-- element: double (containsNull = true)
 |-- timestamp: timestamp (nullable = true)



## Task 3: Clean the DataFrame that contains information about users.

In [0]:
from pyspark.sql.functions import concat_ws, col, to_timestamp

# Create a new column "user_name" by concatenating "first_name" and "last_name"
df_user = df_user.withColumn("user_name", concat_ws(" ", col("first_name"), col("last_name")))

# Drop the "first_name" and "last_name" columns
df_user = df_user.drop("first_name", "last_name")

# Convert the "date_joined" column from string to timestamp
df_user = df_user.withColumn("date_joined", to_timestamp(col("date_joined")))

# Reorder the columns
df_user_cleaned = df_user.select("ind", "user_name", "age", "date_joined")

# Show the cleaned DataFrame schema
print("Schema for df_user_cleaned:")
df_user_cleaned.printSchema()

# Print out the first 5 rows to verify the data (particularly the "user_name" column)
display(df_user_cleaned.head(5))

Schema for df_user_cleaned:
root
 |-- ind: long (nullable = true)
 |-- user_name: string (nullable = false)
 |-- age: long (nullable = true)
 |-- date_joined: timestamp (nullable = true)



ind,user_name,age,date_joined
2814,Christopher Hernandez,38,2015-11-12T11:07:24Z
3587,Christopher Rodriguez,25,2016-11-24T18:36:05Z
8221,Christopher Edwards,24,2016-04-06T21:17:29Z
9582,Christopher Hawkins,45,2016-09-15T07:02:53Z
1683,Christina Carpenter,46,2016-07-02T09:06:40Z


## Write DataFrames to Delta Tables in DBFS

In [0]:
# Define the base path
base_path = "dbfs:/user/hive/warehouse/"

# Write Pinterest data
df_pin_cleaned.write.format("delta").mode("overwrite").save(base_path + "pin_table")

# Write Geolocation data
df_geo_cleaned.write.format("delta").mode("overwrite").save(base_path + "geo_table")

# Write User data
df_user_cleaned.write.format("delta").mode("overwrite").save(base_path + "user_table")

In [0]:
%fs ls /user/hive/warehouse/

path,name,size,modificationTime
dbfs:/user/hive/warehouse/authentication_credentials/,authentication_credentials/,0,1742330574588
dbfs:/user/hive/warehouse/geo_table/,geo_table/,0,1742330574588
dbfs:/user/hive/warehouse/pin_table/,pin_table/,0,1742330574588
dbfs:/user/hive/warehouse/user_table/,user_table/,0,1742330574588


## Task 4: Find the most popular category in each country.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, count, row_number

# Join df_pin_cleaned and df_geo_cleaned on the "ind" column
df_joined = df_pin_cleaned.join(df_geo_cleaned, on="ind", how="inner")

# Group by "country" and "category" and count posts per category
df_category_count = df_joined.groupBy("country", "category").agg(
    count("*").alias("category_count")
)

# Find the maximum category_count for each country using a window function
window_spec = Window.partitionBy("country").orderBy(col("category_count").desc())
df_most_popular = df_category_count.withColumn("rank", row_number().over(window_spec)) \
                                   .filter(col("rank") == 1) \
                                   .drop("rank")

# Show the result
display(df_most_popular)

country,category,category_count
Afghanistan,finance,5
Albania,art,9
Algeria,quotes,12
American Samoa,tattoos,8
Andorra,quotes,3
Angola,education,2
Anguilla,diy-and-crafts,2
Antarctica (the territory South of 60 deg S),christmas,4
Antigua and Barbuda,home-decor,1
Argentina,quotes,3


## Task 5: Find which was the most popular category each year

In [0]:
from pyspark.sql.functions import col, year, count

# Extract the year from the "timestamp" column
df_geo_cleaned = df_geo_cleaned.withColumn("post_year", year(col("timestamp")))

# Filter the data to include only posts between 2018 and 2022
df_geo_cleaned = df_geo_cleaned.filter((col("post_year") >= 2018) & (col("post_year") <= 2022))

# Join df_pin_cleaned and df_geo_cleaned on the "ind" column
df_joined = df_pin_cleaned.join(df_geo_cleaned, on="ind", how="inner")

# Group by "post_year" and "category" and count the number of posts
df_category_count = df_joined.groupBy("post_year", "category").agg(count("*").alias("category_count")).orderBy("post_year")

# Display the result
display(df_category_count)

post_year,category,category_count
2018,mens-fashion,11
2018,christmas,8
2018,finance,14
2018,diy-and-crafts,9
2018,beauty,6
2018,education,7
2018,travel,7
2018,tattoos,10
2018,home-decor,7
2018,art,8


## Task 6: Find the user with most followers in each country

In [0]:
# Step 1: For each country, find the user with the most followers.

# Import necessary libraries
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Join df_pin_cleaned and df_geo_cleaned on the "ind" column to combine user and country information
df_joined = df_pin_cleaned.join(df_geo_cleaned, on="ind", how="inner")

# Define a window specification to rank users by follower_count within each country
# Partition by "country" and order by "follower_count" in descending order
window_spec = Window.partitionBy("country").orderBy(col("follower_count").desc())

# Add a rank column to each row based on the window specification
df_ranked = df_joined.withColumn("rank", row_number().over(window_spec))

# Filter to keep only the top-ranked user (rank = 1) for each country
# Select only the relevant columns: country, poster_name, and follower_count
df_most_followers_by_country = df_ranked.filter(col("rank") == 1).select("country", "poster_name", "follower_count")

# Show the result for Step 1
print("Countries with their user who has the most followers:")
display(df_most_followers_by_country)

# Step 2: Based on the above query, find the country with the user who has the most followers.

# Order the result from Step 1 by follower_count in descending order
# Limit the result to 1 row to get the country with the user who has the most followers
# Select only the relevant columns: country and follower_count
df_country_most_followers = df_most_followers_by_country.orderBy(col("follower_count").desc()).limit(1).select("country", "follower_count")

# Show the result for Step 2
print("Country with the user who has the most followers overall:")
display(df_country_most_followers)

Countries with their user who has the most followers:


country,poster_name,follower_count
Afghanistan,Walmart,2000000
Albania,The Minds Journal,5000000
Algeria,YourTango,942000
American Samoa,Mamas Uncut,8000000
Andorra,Teachers Pay Teachers,1000000
Angola,Pathway 2 Success,56000
Anguilla,"Kristen | Lifestyle, Mom Tips & Teacher Stuff Blog",91000
Antarctica (the territory South of 60 deg S),Refinery29,1000000
Antigua and Barbuda,Country Living Magazine,1000000
Argentina,Cheezburger,2000000


Country with the user who has the most followers overall:


country,follower_count
American Samoa,8000000


## Task 7: Find the most popular category for different age groups

In [0]:
# Create a new column "age_group" based on the "age" column
df_user_cleaned = df_user_cleaned.withColumn(
    "age_group",
    when((col("age") >= 18) & (col("age") <= 24), "18-24")
    .when((col("age") >= 25) & (col("age") <= 35), "25-35")
    .when((col("age") >= 36) & (col("age") <= 50), "36-50")
    .otherwise("+50")
)

# Join df_pin_cleaned and df_user_cleaned on the "ind" column
df_joined = df_pin_cleaned.join(df_user_cleaned, on="ind", how="inner")

# Group by "age_group" and "category" and count the number of posts
df_category_count = df_joined.groupBy("age_group", "category").agg(count("*").alias("category_count")).orderBy("age_group", "category")

# Show the result
print("Category count for each age group:")
display(df_category_count)

# Use a window function to rank categories by count for each age group
window_spec = Window.partitionBy("age_group").orderBy(col("category_count").desc())

df_ranked = df_category_count.withColumn("rank", row_number().over(window_spec))

# Filter to get only the most popular category for each age group
df_most_popular_category_by_age_group = df_ranked.filter(col("rank") == 1).drop("rank")

# Show the result
print("Most popular category for each age group:")
display(df_most_popular_category_by_age_group)

Category count for each age group:


age_group,category,category_count
+50,art,2
+50,beauty,5
+50,christmas,2
+50,diy-and-crafts,4
+50,education,5
+50,event-planning,4
+50,finance,2
+50,home-decor,1
+50,mens-fashion,4
+50,quotes,2


Most popular category for each age group:


age_group,category,category_count
+50,vehicles,7
18-24,tattoos,29
25-35,christmas,15
36-50,christmas,13


## Task 8: Find the median follower count for different age groups

In [0]:
print(spark.version)  # If ≥ 3.1.1, `median()` is supported

3.5.2


In [0]:
from pyspark.sql.functions import col, row_number, median
# Create a new column "age_group" based on the "age" column
df_user_cleaned = df_user_cleaned.withColumn(
    "age_group",
    when((col("age") >= 18) & (col("age") <= 24), "18-24")
    .when((col("age") >= 25) & (col("age") <= 35), "25-35")
    .when((col("age") >= 36) & (col("age") <= 50), "36-50")
    .otherwise("+50")
)

# Join df_pin_cleaned and df_user_cleaned on the "ind" column
df_joined = df_pin_cleaned.join(df_user_cleaned, on="ind", how="inner")

# Group by "age_group" and "category" and calculate the median follower count for each age group
df_follower_count_median_by_age_group  = df_joined.groupBy("age_group").agg(median("follower_count").alias("median_follower_count")).orderBy("age_group")

# Show the result
print("Median follower count for users in the following age groups:")
display(df_follower_count_median_by_age_group)

Median follower count for users in the following age groups:


age_group,median_follower_count
+50,4500.0
18-24,132000.0
25-35,22000.0
36-50,12500.0


## Task 9: Find how many users have joined each year?

In [0]:
from pyspark.sql.functions import col, year, count

# Extract the year from the "date_joined" column
df_user_cleaned = df_user_cleaned.withColumn("joined_year", year(col("date_joined")))

# Filter the data to include only users who joined between 2015 and 2020
df_filtered = df_user_cleaned.filter((col("joined_year") >= 2015) & (col("joined_year") <= 2020))

# # Join df_user_cleaned and df_filtered on the "ind" column
# df_joined = df_user_cleaned.join(df_filtered, on="ind", how="inner")

# Group by "joined_year" and count the number of users who joined each year
df_joined_users_each_year = df_filtered.groupBy("joined_year").agg(count("user_name").alias("number_users_joined")).orderBy("joined_year")

# Show the result
print("Number of users who joined each year:")
display(df_joined_users_each_year)



Number of users who joined each year:


joined_year,number_users_joined
2015,218
2016,201
2017,81


## Task 10: Find the median follower count of users based on their joining year

In [0]:
from pyspark.sql.functions import col, year, count, median

# Extract the year from the "date_joined" column
df_user_cleaned = df_user_cleaned.withColumn("joined_year", year(col("date_joined")))

# Filter the data to include only users who joined between 2015 and 2020
df_filtered = df_user_cleaned.filter((col("joined_year") >= 2015) & (col("joined_year") <= 2020))

# Join df_user_cleaned and df_filtered on the "ind" column
df_joined = df_pin_cleaned.join(df_filtered, on="ind", how="inner")

# Group by "joined_year" and calculate the median follower count for each year
df_median_follower_count_by_year = df_joined.groupBy("joined_year").agg(median("follower_count").alias("median_follower_count")).orderBy("joined_year")

# Show the result
print("Median follower count for each year:")
display(df_median_follower_count_by_year)



Median follower count for each year:


joined_year,median_follower_count
2015,117000.0
2016,23000.0
2017,4000.0


## Task 11: Find the median follower count of users based on their joining year and age group

In [0]:
from pyspark.sql.functions import col, median

# Join df_pin_cleaned and df_user_cleaned on the "ind" column
df_joined = df_pin_cleaned.join(df_user_cleaned, on="ind", how="inner")


from pyspark.sql.functions import col, year, count, median

# Create a new column "age_group" based on the "age" column
df_user_cleaned = df_user_cleaned.withColumn(
    "age_group",
    when((col("age") >= 18) & (col("age") <= 24), "18-24")
    .when((col("age") >= 25) & (col("age") <= 35), "25-35")
    .when((col("age") >= 36) & (col("age") <= 50), "36-50")
    .otherwise("+50")
)

# Extract the year from the "timestamp" column
df_geo_cleaned = df_geo_cleaned.withColumn("post_year", year(col("timestamp")))

# Filter the data to include only users who joined between 2015 and 2020
df_filtered = df_geo_cleaned.filter((col("post_year") >= 2015) & (col("post_year") <= 2020))

# Join df_pin_cleaned, df_user_cleaned and df_filtered on the "ind" column
df_joined = df_pin_cleaned.join(df_filtered, on="ind", how="inner").join(df_user_cleaned, on="ind", how="inner")

# Group by "age_group", post_year" and calculate the median follower count for each year
df_median_follower_count_each_year_by_age_group = df_joined.groupBy("age_group", "post_year").agg(median("follower_count").alias("median_follower_count")).orderBy("age_group", "post_year")

# Show the result
print("Median follower count for each year by age group:")
display(df_median_follower_count_each_year_by_age_group)



Median follower count for each year by age group:


age_group,post_year,median_follower_count
+50,2018,7500.0
+50,2019,586.5
+50,2020,9000.0
18-24,2018,109500.0
18-24,2019,104000.0
18-24,2020,117000.0
25-35,2018,22000.0
25-35,2019,15000.0
25-35,2020,17000.0
36-50,2018,5000.0


# Milestone 7: Batch Processing: AWS MWAA

Once the connection between MWAA and Databricks is setup and DAG is unpaused, you can go to Databricks -> `Job Runs` to inspect the job status.

## Set Up the Databricks Connection

1. Ensure you have the apache-airflow-providers-databricks package installed:
    ```
    pip install apache-airflow-providers-databricks
    ```

2. Set Up the Databricks Connection in Airflow UI
    1. Open the Airflow UI:
        
        Go to **Admin** → **Connections**.
        
    2. Create a New Connection:
      
        Click the + button to add a new connection.
        
    3. Fill in the Connection Details:
    
        **Conn Id**: databricks_default (or any custom name you prefer).
        
        **Conn Type**: Select Databricks.
        
        **Host**: Your Databricks workspace URL (e.g., https://<databricks-instance>.cloud.databricks.com).
        
        **Token**: Your Databricks personal access token.
        
        _To generate a token: Go to your Databricks workspace. Click on your user profile → User Settings → Developer → Access Tokens → Generate New Token._
        
        **Extra** (optional): Add additional parameters if needed, such as:
        ```
          {
            "cluster_id": "your-cluster-id",
            "region": "your-region"
          }
        ```
    4. Save the Connection.

3. Verify the Connection

    Check the Connection in the UI:
    Go back to Admin → Connections and verify that the connection is listed.

4. Use the Connection in a DAG

    Here’s an example of how to use the Databricks connection in a DAG:
    ```
    from airflow import DAG
    from airflow.providers.databricks.operators.databricks import DatabricksSubmitRunOperator
    from datetime import datetime

    # Define the DAG
    with DAG(
        dag_id="databricks_example",
        start_date=datetime(2023, 1, 1),
        schedule_interval="@daily",
        catchup=False,
    ) as dag:

      # Define the Databricks task
      databricks_task = DatabricksSubmitRunOperator(
          task_id="databricks_task",
          databricks_conn_id="databricks_default",  # Use the connection ID
          new_cluster={
              "spark_version": "10.4.x-scala2.12",
              "node_type_id": "i3.xlarge",
              "num_workers": 2,
          },
          notebook_task={
              "notebook_path": "/Users/your-username/your-notebook",
          },
      )

      # Set task dependencies (if any)
      databricks_task
    ```


Restart airflow:
```
pkill -f "airflow scheduler"
pkill -f "airflow webserver"
airflow scheduler &
airflow webserver &
```

# Milestone 8: Stream Processing: AWS Kinesis

## Task 4: Read data from Kinesis streams in Databricks

In [0]:
# Execute the SQL query and store the result in a variable named 'authentication_credentials'
aws_keys_df = spark.sql("""
SELECT *
FROM pinterest_data_engineering_project_3969908457353693.default.authentication_credentials
""")

In [0]:
import urllib  # Import the urllib module for URL handling

# Retrieve the AWS Access Key ID from the DataFrame 'aws_keys_df'
ACCESS_KEY = aws_keys_df.select('Access key ID').collect()[0]['Access key ID']
# Retrieve the AWS Secret Access Key from the DataFrame 'aws_keys_df'
SECRET_KEY = aws_keys_df.select('Secret access key').collect()[0]['Secret access key']
# URL-encode the Secret Access Key to ensure it is safe for use in URLs
ENCODED_SECRET_KEY = urllib.parse.quote(string=SECRET_KEY, safe="")

In [0]:
# Create a streaming DataFrame by reading from an AWS Kinesis stream
df = spark \
.readStream \
.format('kinesis') \
.option('streamName','Kinesis-Prod-Stream') \
.option('initialPosition','earliest') \
.option('region','eu-west-1') \
.option('awsAccessKey', ACCESS_KEY) \
.option('awsSecretKey', SECRET_KEY) \
.load()

In [0]:
display(df)

partitionKey,data,stream,shardId,sequenceNumber,approximateArrivalTimestamp
geolocation_data,eyJpbmQiOiAyMiwgInRpbWVzdGFtcCI6ICIyMDE4LTEyLTAyVDEyOjEyOjAwIiwgImxhdGl0dWRlIjogLTg2LjI5MTksICJsb25naXR1ZGUiOiAtMTAzLjk5MywgImNvdW50cnkiOiAiQW5ndWlsbGEifQ==,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695724089717838259411620986882,2025-03-18T20:24:12.273Z
geolocation_data,eyJpbmQiOiAzNiwgInRpbWVzdGFtcCI6ICIyMDE4LTAxLTIwVDA4OjMxOjUwIiwgImxhdGl0dWRlIjogNjcuNzg1NSwgImxvbmdpdHVkZSI6IDExNS4wNDMsICJjb3VudHJ5IjogIkxpdGh1YW5pYSJ9,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695725298643657874040795693058,2025-03-18T20:24:12.273Z
geolocation_data,eyJpbmQiOiA1OCwgInRpbWVzdGFtcCI6ICIyMDE4LTA1LTE2VDIwOjMxOjU0IiwgImxhdGl0dWRlIjogNTAuNzkxNiwgImxvbmdpdHVkZSI6IDE1NS41NzksICJjb3VudHJ5IjogIk1hbGRpdmVzIn0=,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695726507569477488669970399234,2025-03-18T20:24:12.273Z
geolocation_data,eyJpbmQiOiA2MCwgInRpbWVzdGFtcCI6ICIyMDIyLTA1LTI3VDExOjA2OjA4IiwgImxhdGl0dWRlIjogLTMxLjI4OTgsICJsb25naXR1ZGUiOiAxMTAuNjUsICJjb3VudHJ5IjogIkFudGFyY3RpY2EgKHRoZSB0ZXJyaXRvcnk= (truncated),Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695727716495297103299145105410,2025-03-18T20:24:12.273Z
geolocation_data,eyJpbmQiOiA3MiwgInRpbWVzdGFtcCI6ICIyMDIyLTA2LTAxVDA5OjIxOjU5IiwgImxhdGl0dWRlIjogLTg2LjQ0MzMsICJsb25naXR1ZGUiOiAtMTc4Ljc3MiwgImNvdW50cnkiOiAiQW1lcmljYW4gU2Ftb2EifQ==,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695728925421116717928319811586,2025-03-18T20:24:12.273Z
geolocation_data,eyJpbmQiOiAxMjYsICJ0aW1lc3RhbXAiOiAiMjAxOS0wOC0xNlQwOTo1NDoxNCIsICJsYXRpdHVkZSI6IC03MS42ODU2LCAibG9uZ2l0dWRlIjogLTE3OS4xMjYsICJjb3VudHJ5IjogIkFsYmFuaWEifQ==,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695730134346936332557494517762,2025-03-18T20:24:12.275Z
geolocation_data,eyJpbmQiOiAxNDksICJ0aW1lc3RhbXAiOiAiMjAxOC0wNS0yN1QwNzozMTo1NSIsICJsYXRpdHVkZSI6IC04Ny4wNTc0LCAibG9uZ2l0dWRlIjogLTE2NC44MjYsICJjb3VudHJ5IjogIkJhaGFtYXMifQ==,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695731343272755947186669223938,2025-03-18T20:24:12.275Z
geolocation_data,eyJpbmQiOiAxOTEsICJ0aW1lc3RhbXAiOiAiMjAyMC0wMS0xMlQwNzowNTowMyIsICJsYXRpdHVkZSI6IC04Ni40NDMzLCAibG9uZ2l0dWRlIjogLTE3OC43NzIsICJjb3VudHJ5IjogIkFtZXJpY2FuIFNhbW9hIn0=,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695732552198575561815843930114,2025-03-18T20:24:12.275Z
geolocation_data,eyJpbmQiOiAxOTksICJ0aW1lc3RhbXAiOiAiMjAyMC0wMi0xNlQxMzoyNToyMCIsICJsYXRpdHVkZSI6IC00NC42MzgxLCAibG9uZ2l0dWRlIjogODIuMTI0LCAiY291bnRyeSI6ICJCb2xpdmlhIn0=,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695733761124395176445018636290,2025-03-18T20:24:12.275Z
geolocation_data,eyJpbmQiOiAyMTcsICJ0aW1lc3RhbXAiOiAiMjAyMC0wNy0yN1QwMDoxMToxMCIsICJsYXRpdHVkZSI6IC04OS4zNTY4LCAibG9uZ2l0dWRlIjogLTM1Ljg1MywgImNvdW50cnkiOiAiQ29uZ28ifQ==,Kinesis-Prod-Stream,shardId-000000000000,49661521598491590166167601695734970050214791074193342466,2025-03-18T20:24:12.275Z


In [0]:
# Filter the streaming DataFrame to only include records with a specific partition key
# This helps in processing only relevant data for our use case

# Filter records where the partition key is "pinterest_data"
df_pin = df.filter(df.partitionKey == "pinterest_data")
# Filter records where the partition key is "geolocation_data"
df_geo = df.filter(df.partitionKey == "geolocation_data")
# Filter records where the partition key is "user_data"
df_user = df.filter(df.partitionKey == "user_data")

In [0]:
# Deserialize the 'data' column of the DataFrame and alias it as 'jsonData'
# This converts the binary data into a readable string format for further processing

# Convert the 'data' column to a STRING type and alias it as 'jsonData' for the Pinterest data
df_pin = df_pin.selectExpr("CAST(data as STRING) jsonData")
# Convert the 'data' column to a STRING type and alias it as 'jsonData' for the geolocation data
df_geo = df_geo.selectExpr("CAST(data as STRING) jsonData")
# Convert the 'data' column to a STRING type and alias it as 'jsonData' for the user data
df_user = df_user.selectExpr("CAST(data as STRING) jsonData")

In [0]:
display(df_pin)
display(df_geo)
display(df_user)

jsonData
"{""ind"": 22, ""timestamp"": ""2018-12-02T12:12:00"", ""latitude"": -86.2919, ""longitude"": -103.993, ""country"": ""Anguilla""}"
"{""ind"": 36, ""timestamp"": ""2018-01-20T08:31:50"", ""latitude"": 67.7855, ""longitude"": 115.043, ""country"": ""Lithuania""}"
"{""ind"": 58, ""timestamp"": ""2018-05-16T20:31:54"", ""latitude"": 50.7916, ""longitude"": 155.579, ""country"": ""Maldives""}"
"{""ind"": 60, ""timestamp"": ""2022-05-27T11:06:08"", ""latitude"": -31.2898, ""longitude"": 110.65, ""country"": ""Antarctica (the territory South of 60 deg S)""}"
"{""ind"": 72, ""timestamp"": ""2022-06-01T09:21:59"", ""latitude"": -86.4433, ""longitude"": -178.772, ""country"": ""American Samoa""}"
"{""ind"": 126, ""timestamp"": ""2019-08-16T09:54:14"", ""latitude"": -71.6856, ""longitude"": -179.126, ""country"": ""Albania""}"
"{""ind"": 149, ""timestamp"": ""2018-05-27T07:31:55"", ""latitude"": -87.0574, ""longitude"": -164.826, ""country"": ""Bahamas""}"
"{""ind"": 191, ""timestamp"": ""2020-01-12T07:05:03"", ""latitude"": -86.4433, ""longitude"": -178.772, ""country"": ""American Samoa""}"
"{""ind"": 199, ""timestamp"": ""2020-02-16T13:25:20"", ""latitude"": -44.6381, ""longitude"": 82.124, ""country"": ""Bolivia""}"
"{""ind"": 217, ""timestamp"": ""2020-07-27T00:11:10"", ""latitude"": -89.3568, ""longitude"": -35.853, ""country"": ""Congo""}"


In [0]:
# By aliasing this column to data and selecting all columns from the MapType select("data.*") we can view each json as column seperated values
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType  # Add relevant data types

# Define your JSON schema structure for Pinterest data, Geolocation data and User data
struct_pin = StructType([
    StructField("category", StringType()),
    StructField("description", StringType()),
    StructField("downloaded", LongType()),
    StructField("follower_count", StringType()),
    StructField("image_src", StringType()),
    StructField("index", LongType()),
    StructField("is_image_or_video", StringType()),
    StructField("poster_name", StringType()),
    StructField("save_location", StringType()),
    StructField("tag_list", StringType()),
    StructField("title", StringType()),
    StructField("unique_id", StringType())
    ])
struct_geo = StructType([
    StructField("ind", LongType()),    
    StructField("timestamp", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("country", StringType())
])
struct_user = StructType([
    StructField("ind", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("age", LongType()),
    StructField("date_joined", StringType())
    ])

# Parse the JSON column for Pinterest data, Geolocation data and User data
df_geo = df_geo.select(
    from_json("jsonData", struct_geo).alias("data")
).select("data.*")
df_pin = df_pin.select(
    from_json("jsonData", struct_pin).alias("data")
).select("data.*")
df_user = df_user.select(
    from_json("jsonData", struct_user).alias("data")
).select("data.*")


In [0]:
# Check data has been parsed correctly
display(df_pin)

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
art,"My first successful one-point perspective art lesson that my students loved. Perspective is part magic and part math and for creative types like me, there",1,79k,https://i.pinimg.com/originals/8a/2d/63/8a2d634787085182deed30aca0914745.png,22,image,DEEP SPACE SPARKLE,Local save in /data/art,"Art Lessons For Kids,Art Lessons Elementary,Art For Kids,Kids Drawing Lessons,Drawing Ideas Kids,Elementary Schools,Elementary Drawing,Kids Art Class,Classroom Art Projects",One-Point Perspective Art Lesson | Deep Space Sparkle,aa1d63a9-1c9e-46b9-9c8a-2379df55a406
art,"Forever Fern Hello Card inspiration taken from Pinterest. Instead of going for all greens, I've used Magenta Madness and Gold in the colour combination.",1,11k,https://i.pinimg.com/originals/9a/05/d1/9a05d131799c17d14057fb4375187db0.jpg,36,image,Alisa Tilsner Stampin' Up! Demonstrator Australia,Local save in /data/art,"Diy Quote Cards,Cards Diy,Art Cards,Painting Inspiration,Art Inspo,Inspiration Quotes,Watercolor Cards,Watercolor Paintings,Quote Paintings",Forever Fern Hello Card - Alisa Tilsner,089f6b5f-726a-42c1-8a95-366b07f5f749
art,So kannst du aus natürlichen Materialien schöne Duftkerzen selber machen. Als Geschenk zu Weihnachten oder als Deko für dein Zuhause.,1,40k,https://i.pinimg.com/originals/d1/ad/1d/d1ad1d228decd9e55f274f0e00138179.jpg,58,image,Alva & Ida - DIY & Nachhaltigkeit,Local save in /data/art,"Line Art Design,Minimal Art,Kunst Tattoos,Line Art Tattoos,Outline Art,Colossal Art,Abstract Line Art,Diy Canvas Art,Art Drawings Sketches",DIY Duftkerzen mit Orange und Rosmarin selber machen,41006daf-2f1c-445a-a06c-d4ccd420be9c
art,Tropical Leaf Free Printable Art -Series of 9 | The Happy Housie | Beautiful free summer printables with customizable options and easy to download for your summer home decor. #s…,1,193k,https://i.pinimg.com/originals/e5/0f/17/e50f176ab830b2e89b9f1a2c2cdb562d.jpg,60,image,The Happy Housie,Local save in /data/art,"Tropical Leaves,Tropical Plants,Tropical Decor,Cactus Plants,Tropical Colors,Flowering Plants,Summer Plants,Tropical Flower Arrangements,Tropical Interior",Tropical Leaf Free Printable Art {Series of 9} | The Happy Housie,2e4bcf46-60ef-4b38-946f-72feb77be12c
art,67 Surreal Castle Concept Art Depictions to Surge Inspiration From #concept #fortress #medieval #cathedral #castle #conceptart #concept,1,556k,https://i.pinimg.com/originals/45/af/a6/45afa61456556b2b8304ade5173f13d9.jpg,72,image,Homesthetics.net,Local save in /data/art,"Dark Fantasy Art,Fantasy Kunst,Fantasy Concept Art,Fantasy City,Fantasy Castle,Fantasy Places,Fantasy Artwork,Gothic Castle,Dark Castle","67 Fantasy And Medieval Buildings, Cities & Castles Concept Art To Inspire You | Homesthetics - Inspiring Ideas For Your Home.",cfe4bdea-8976-40e5-8c1b-e2850206b189
art,Create beautiful mixed media winter art with easy techniques and simple supplies. A fun winter art project that kids will love to create!,1,20k,https://i.pinimg.com/originals/00/2d/bd/002dbd8cbb5ae07950bc704e29e09084.jpg,126,image,Projects with Kids,Local save in /data/art,"Kindergarten Art Projects,Classroom Art Projects,School Art Projects,Art Classroom,Winter Art Kindergarten,Art Projects For Kindergarteners,Christmas Art Projects,Winter Art Projects,Easy Kids Art Projects",Mixed Media Winter Art Project for Kids,0e144db9-7e3f-4d7c-94ea-b9f5b28b2661
art,These fall art projects showcase the best the season has to offer. These simple fall crafts use simeple materials to take advantage of nature's beauty!,1,221k,https://i.pinimg.com/originals/4f/a7/55/4fa75527fe7de2dac148e006f8f401fe.png,149,image,The Kitchen Table Classroom,Local save in /data/art,"Fall Paper Crafts,Fall Crafts For Kids,Arts And Crafts,Art Crafts,Nature Crafts,Autumn Crafts,Summer Crafts,Easter Crafts,Paper Crafting",Fall Art Projects-Fall Crafts from Nature - The Kitchen Table Classroom,35194432-35f

In [0]:
display(df_geo)

ind,timestamp,latitude,longitude,country
22,2018-12-02T12:12:00,-86.2919,-103.993,Anguilla
36,2018-01-20T08:31:50,67.7855,115.043,Lithuania
58,2018-05-16T20:31:54,50.7916,155.579,Maldives
60,2022-05-27T11:06:08,-31.2898,110.65,Antarctica (the territory South of 60 deg S)
72,2022-06-01T09:21:59,-86.4433,-178.772,American Samoa
126,2019-08-16T09:54:14,-71.6856,-179.126,Albania
149,2018-05-27T07:31:55,-87.0574,-164.826,Bahamas
191,2020-01-12T07:05:03,-86.4433,-178.772,American Samoa
199,2020-02-16T13:25:20,-44.6381,82.124,Bolivia
217,2020-07-27T00:11:10,-89.3568,-35.853,Congo


In [0]:
display(df_user)

ind,first_name,last_name,age,date_joined
22,Abigail,Bishop,24,2015-12-28T14:42:33
36,Julie,Thompson,58,2016-11-16T02:11:02
58,Marie,Rivas,22,2016-12-09T14:12:57
60,Charles,Hutchinson,31,2016-01-17T23:28:43
72,Alexandra,Allen,20,2015-10-23T14:07:00
126,Aaron,Bartlett,21,2015-11-24T02:15:36
149,Andrew,Burke,20,2015-11-14T17:38:31
191,Alexandra,Allen,20,2015-10-23T14:07:00
199,Cynthia,Harper,43,2016-01-23T05:44:11
217,Anthony,Martinez,20,2015-11-29T09:31:00


## Task 5: Transform Kinesis streams in Databricks

In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace, expr, array, to_timestamp, concat_ws

# Task 1: Clean the DataFrame containing Pinterest post information

# Replace empty or irrelevant string entries with None
string_columns = [col_name for col_name, dtype in df_pin.dtypes if dtype == 'string']
for column in string_columns:
    df_pin = df_pin.withColumn(
        column, 
        when(trim(col(column)) != "", col(column)).otherwise(None)
    )

# Replace invalid or missing numeric entries with None
numeric_columns = [col_name for col_name, dtype in df_pin.dtypes if dtype in ['int', 'double', 'long']]
for column in numeric_columns:
    df_pin = df_pin.withColumn(
        column, 
        when(col(column).isNotNull(), col(column)).otherwise(None)
    )

# Convert follower_count to integer, handling suffixes (k, M, B)
df_pin = df_pin.withColumn(
    "follower_count",
    regexp_replace(
        regexp_replace(
            regexp_replace(col("follower_count"), "(?i)k$", "000"),     # 1k -> 1000
            "(?i)m$", "000000"                                          # 1M -> 1000000
        ),
        "(?i)b$", "000000000"                                           # 1B -> 1000000000
    )
)

# Identify invalid follower_count entries
df_invalid_entries = df_pin.filter(~col("follower_count").rlike(r"^\d+$"))
# print("Invalid follower_count entries:")
# display(df_invalid_entries)

# Convert follower_count to integer, handling errors
df_pin = df_pin.withColumn("follower_count", expr("try_cast(follower_count as int)"))

# Ensure numeric columns are correctly typed
numeric_columns = ["downloaded", "follower_count", "index"]
for column in numeric_columns:
    df_pin = df_pin.withColumn(column, col(column).cast("int"))

# Clean save_location by removing "Local save in " prefix
df_pin = df_pin.withColumn("save_location", regexp_replace(col("save_location"), "^Local save in ", ""))

# Rename index column
df_pin = df_pin.withColumnRenamed("index", "ind")

# Reorder columns
desired_order = [
    "ind", "unique_id", "title", "description", "follower_count", 
    "poster_name", "tag_list", "is_image_or_video", 
    "image_src", "save_location", "category"
]
df_pin_cleaned = df_pin.select(desired_order)

# Display cleaned data and schema
# display(df_pin_cleaned.head(5))
# print(f'The number of rows after cleaning df_pin: {df_pin_cleaned.count()}')
print("Schema for df_pin_cleaned:")
df_pin_cleaned.printSchema()

# Task 2: Clean the DataFrame containing geolocation information

# Create a coordinates column and drop latitude/longitude
df_geo = df_geo.withColumn("coordinates", array(col("latitude"), col("longitude")))
df_geo = df_geo.drop("latitude", "longitude")

# Convert timestamp column to timestamp type
df_geo = df_geo.withColumn("timestamp", to_timestamp(col("timestamp")))

# Reorder columns
df_geo_cleaned = df_geo.select("ind", "country", "coordinates", "timestamp")

# Display cleaned geolocation data and schema
# display(df_geo_cleaned.head(5))
print("Schema for df_geo_cleaned:")
df_geo_cleaned.printSchema()

# Task 3: Clean the DataFrame containing user information

# Create user_name column by concatenating first and last name
df_user = df_user.withColumn("user_name", concat_ws(" ", col("first_name"), col("last_name")))

# Drop first_name and last_name columns
df_user = df_user.drop("first_name", "last_name")

# Convert date_joined to timestamp type
df_user = df_user.withColumn("date_joined", to_timestamp(col("date_joined")))

# Reorder columns
df_user_cleaned = df_user.select("ind", "user_name", "age", "date_joined")

# Display cleaned user data and schema
# display(df_user_cleaned.head(5))
print("Schema for df_user_cleaned:")
df_user_cleaned.printSchema()

Schema for df_pin_cleaned:
root
 |-- ind: integer (nullable = true)
 |-- unique_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- follower_count: integer (nullable = true)
 |-- poster_name: string (nullable = true)
 |-- tag_list: string (nullable = true)
 |-- is_image_or_video: string (nullable = true)
 |-- image_src: string (nullable = true)
 |-- save_location: string (nullable = true)
 |-- category: string (nullable = true)

Schema for df_geo_cleaned:
root
 |-- ind: long (nullable = true)
 |-- country: string (nullable = true)
 |-- coordinates: array (nullable = false)
 |    |-- element: double (containsNull = true)
 |-- timestamp: timestamp (nullable = true)

Schema for df_user_cleaned:
root
 |-- ind: long (nullable = true)
 |-- user_name: string (nullable = false)
 |-- age: long (nullable = true)
 |-- date_joined: timestamp (nullable = true)



## Task 6: Write the streaming data to Delta Tables

See [Delta table streaming reads and writes](https://docs.databricks.com/aws/en/structured-streaming/delta-lake?language=Python)

**[Each query must have a different checkpoint location. Multiple queries should never share the same location.](https://docs.databricks.com/aws/en/structured-streaming/checkpoints)**

Please note each `writeStream` call should be manually stopped once all the data has been stored in a Delta table (when Input vs. Processing Rate is 0), Otherwise it will run indefinitely.

All of the following code should be ran on a all purpose compute cluster as `writeStream` will give error if it's ran on a SQL warehouse serverless compute.

In [0]:
# Remove the existing checkpoint directory and all its contents from the specified path
# The 'True' parameter indicates that the removal should be recursive
dbutils.fs.rm("/tmp/kinesis/_checkpoints_pin/", True)

# Create a new checkpoint directory at the specified path
# This directory will be used to store checkpoint data for streaming operations
dbutils.fs.mkdirs("/tmp/kinesis/_checkpoints_pin")

True

In [0]:
# Write the cleaned Pinterest data to a Delta table in append mode
df_pin_cleaned.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "/tmp/kinesis/_checkpoints_pin/") \
  .table("808492447622_pin_table")

# df_pin_cleaned.awaitTermination()


In [0]:
# Remove the existing checkpoint directory and all its contents from the specified path
# The 'True' parameter indicates that the removal should be recursive
dbutils.fs.rm("/tmp/kinesis/_checkpoints_geo/", True)

# Create a new checkpoint directory at the specified path
# This directory will be used to store checkpoint data for streaming operations
dbutils.fs.mkdirs("/tmp/kinesis/_checkpoints_geo")

True

In [0]:
# Write the cleaned Geolocation data to a Delta table in append mode
df_geo_cleaned.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "/tmp/kinesis/_checkpoints_geo/") \
  .table("808492447622_geo_table")

In [0]:
# Remove the existing checkpoint directory and all its contents from the specified path
# The 'True' parameter indicates that the removal should be recursive
dbutils.fs.rm("/tmp/kinesis/_checkpoints_user/", True)

# Create a new checkpoint directory at the specified path
# This directory will be used to store checkpoint data for streaming operations
dbutils.fs.mkdirs("/tmp/kinesis/_checkpoints_user")

True

In [0]:
# Write the cleaned User data to a Delta table in append mode
df_user_cleaned.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "/tmp/kinesis/_checkpoints_user/") \
  .table("808492447622_user_table")

## Verify the tables stored in delta lake

In [0]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("DeltaTableRead").getOrCreate()

# Read the Pinterest data
df_pin = spark.read.format("delta").table("808492447622_pin_table")
pin_count = df_pin.count()
print(f"Number of rows in pin table: {pin_count}")

# Read the Geolocation data
df_geo = spark.read.format("delta").table("808492447622_geo_table")
geo_count = df_geo.count()
print(f"Number of rows in geo table: {geo_count}")

# Read the User data
df_user = spark.read.format("delta").table("808492447622_user_table")
user_count = df_user.count()
print(f"Number of rows in user table: {user_count}")

Number of rows in pin table: 500
Number of rows in geo table: 500
Number of rows in user table: 500
